# Plant Disease Competition — Colab Notebook

Run cells **top to bottom**. Models save to Google Drive so you survive session disconnects.

**Runtime**: Runtime → Change runtime type → **T4 GPU** (free) or **A100** (Colab Pro)

---

## Cell 1 — Mount Google Drive
This keeps your models safe if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/plant_competition'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/siglip2-model', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/dinov2-model', exist_ok=True)
print(f'Drive ready at: {DRIVE_DIR}')

## Cell 2 — Install dependencies

In [ ]:
!pip install transformers datasets huggingface_hub accelerate scikit-learn pillow timm hf_transfer -q
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # fast Rust-based downloader, no rate limits
print('Done.')

## Cell 3 — HuggingFace login
Paste your HF **read** token when prompted. Get one at huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login
login()  # paste token when prompted

## Cell 4 — Unzip data from Google Drive
Your Drive folder: `wetransfer_smells-like-ai-spirit_2026-03-26_1327/`

In [ ]:
import zipfile, os

WETRANSFER = '/content/drive/MyDrive/wetransfer_smells-like-ai-spirit_2026-03-26_1327'

print('Unzipping train...')
with zipfile.ZipFile(f'{WETRANSFER}/plant-disease-train.zip', 'r') as z:
    z.extractall('/content/data')

print('Unzipping test...')
with zipfile.ZipFile(f'{WETRANSFER}/plant-disease-test.zip', 'r') as z:
    z.extractall('/content/data')

# Verify — filter out hidden files/junk (.DS_Store, .cache, __MACOSX etc)
train_classes = [
    d for d in os.listdir('/content/data/train')
    if os.path.isdir(f'/content/data/train/{d}') and not d.startswith('.')
]
print(f'\nTrain classes found: {len(train_classes)}')  # should be 39

from pathlib import Path
test_images = list(Path('/content/data/test').rglob('*.jpg')) + list(Path('/content/data/test').rglob('*.JPG'))
print(f'Test images found: {len(test_images)}')  # should be 10976

print('\nData ready.')

## Cell 5 — Clone competition pack from GitHub

In [ ]:
import os

# Replace with your actual GitHub repo URL
GITHUB_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git'

!git clone {GITHUB_URL} /content/competition_pack

# Find where the scripts are (handles nested folders)
PACK = None
for candidate in [
    '/content/competition_pack',
    '/content/competition_pack/competition_pack',
]:
    if os.path.exists(f'{candidate}/advanced/train_vit_large.py'):
        PACK = candidate
        break

if PACK is None:
    raise RuntimeError('Could not find train_vit_large.py — check your repo structure')

print(f'Competition pack found at: {PACK}')
os.chdir(f'{PACK}/advanced')
print('Working directory:', os.getcwd())

## Cell 6 — Detect GPU
Sets batch sizes automatically based on what Colab gave you.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Go to Runtime → Change runtime type → T4 GPU')

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}')
print(f'VRAM: {vram_gb:.0f} GB')

# Batch sizes per GPU
if vram_gb >= 70:   # A100 80GB
    SIGLIP_BATCH, DINOV2_BATCH = 64, 32
elif vram_gb >= 35: # A100 40GB
    SIGLIP_BATCH, DINOV2_BATCH = 32, 16
else:               # T4 15GB (free tier)
    SIGLIP_BATCH, DINOV2_BATCH = 8, 8

print(f'SigLIP2 batch size: {SIGLIP_BATCH}')
print(f'DINOv2 batch size:  {DINOV2_BATCH}')

## Cell 7 — SUBMISSION 1: CLIP zero-shot
**Run this first. Gives you ~65-75% as a safety net while the real model trains.**

In [ ]:
!python clip_zero_shot.py \
  --test-dir /content/data/test \
  --output-csv /content/submission_clip.csv

!python ../final_submission_check.py \
  --csv /content/submission_clip.csv \
  --test-dir /content/data/test

# Download it
import shutil
shutil.copy('/content/submission_clip.csv', '/content/submission.csv')
files.download('/content/submission.csv')
print('>>> SUBMIT THIS NOW on the competition page <<<')

## Cell 8 — MAIN MODEL: Train SigLIP 2

**Best model for this task.** Model saves to Google Drive every epoch — safe against disconnects.

- T4 free: ~3-4h for full dataset. Use `--max-images 15000` if time is short.
- A100: ~50-60 min for full dataset.

In [ ]:
import os

# Use full dataset on A100, cap to 15k on T4 to stay within session time
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
max_images = 0 if vram_gb >= 35 else 15000
print(f'Max images: {"all" if max_images == 0 else max_images}')

!python train_vit_large.py \
  --model google/siglip2-so400m-patch14-384 \
  --data-dir /content/data/train \
  --max-images {max_images} \
  --batch-size {SIGLIP_BATCH} \
  --eval-batch-size {SIGLIP_BATCH * 2} \
  --epochs 5 \
  --lr 1e-5 \
  --weighted-loss \
  --no-phase1 \
  --output-dir {DRIVE_DIR}/siglip2-model

print('SigLIP 2 training done. Model saved to Google Drive.')

## Cell 8b — Fallback: if SigLIP 2 is not on HuggingFace yet
Only run this if Cell 8 fails with a model-not-found error.

In [ ]:
# FALLBACK ONLY — run if Cell 8 fails
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
max_images = 0 if vram_gb >= 35 else 15000

!python train_vit_large.py \
  --model google/vit-large-patch16-224 \
  --data-dir /content/data/train \
  --max-images {max_images} \
  --batch-size {SIGLIP_BATCH} \
  --eval-batch-size {SIGLIP_BATCH * 2} \
  --epochs 5 \
  --lr 2e-5 \
  --weighted-loss \
  --no-phase1 \
  --output-dir {DRIVE_DIR}/siglip2-model

print('ViT-large training done.')

## Cell 9 — SUBMISSION 2: SigLIP 2 TTA + OOD protection

In [ ]:
!python predict_tta.py \
  --model-dir {DRIVE_DIR}/siglip2-model/best \
  --test-dir /content/data/test \
  --output-csv /content/submission_siglip2.csv \
  --num-tta 6 \
  --ood-threshold 0.5

!python ../final_submission_check.py \
  --csv /content/submission_siglip2.csv \
  --test-dir /content/data/test

import shutil
shutil.copy('/content/submission_siglip2.csv', '/content/submission.csv')
shutil.copy('/content/submission_siglip2.csv', f'{DRIVE_DIR}/submission_siglip2.csv')
files.download('/content/submission.csv')
print('>>> SUBMIT THIS NOW <<<')

## Cell 10 — SECOND MODEL: Train DINOv2
For the ensemble. Start this right after submitting Cell 9.

In [ ]:
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
max_images = 0 if vram_gb >= 35 else 15000

!python train_dinov2.py \
  --model facebook/dinov2-large \
  --data-dir /content/data/train \
  --max-images {max_images} \
  --batch-size {DINOV2_BATCH} \
  --epochs 5 \
  --lr 5e-6 \
  --grad-accum 2 \
  --output-dir {DRIVE_DIR}/dinov2-model

print('DINOv2 training done. Model saved to Google Drive.')

## Cell 11 — SUBMISSION 3: Ensemble SigLIP 2 + DINOv2

In [ ]:
!python ensemble_predict.py \
  --model-dirs {DRIVE_DIR}/siglip2-model/best {DRIVE_DIR}/dinov2-model/best \
  --model-types hf dinov2 \
  --weights 0.6 0.4 \
  --test-dir /content/data/test \
  --output-csv /content/submission_ensemble.csv \
  --ood-threshold 0.5

!python ../final_submission_check.py \
  --csv /content/submission_ensemble.csv \
  --test-dir /content/data/test

import shutil
shutil.copy('/content/submission_ensemble.csv', '/content/submission.csv')
shutil.copy('/content/submission_ensemble.csv', f'{DRIVE_DIR}/submission_ensemble.csv')
files.download('/content/submission.csv')
print('>>> SUBMIT THIS NOW — this is your best shot <<<')

## Cell 12 — Emergency: re-download submission from Drive
If your session died and you need your submission file back.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/plant_competition'

import os
for f in ['submission_ensemble.csv', 'submission_siglip2.csv']:
    path = f'{DRIVE_DIR}/{f}'
    if os.path.exists(path):
        print(f'Found: {f}')
        files.download(path)
    else:
        print(f'Not found: {f}')